In [0]:

# Project configuration
catalog_name = "adv_dev_westus3_demo_001"
schema_name = "ecommerce_01"

target_schema = f"{catalog_name}.{schema_name}"

print("SCD Type 2 setup completed.")
print("Target schema:", target_schema)


SCD Type 2 setup completed.
Target schema: adv_dev_westus3_demo_001.ecommerce_01


In [0]:

# Create a parameter to select the dimension
dbutils.widgets.text("dimension_name", "customers", "Dimension name")

dimension_name = dbutils.widgets.get("dimension_name").strip().lower()

print("Selected dimension:", dimension_name)


Selected dimension: products


In [0]:

from pyspark.sql.functions import current_timestamp, lit

# Read the current customer dimension
if dimension_name == "customers":
    source_df = spark.table(
        f"{target_schema}.gold_dim_customers"
    )
elif dimension_name == "products":
    source_df = spark.table(
        f"{target_schema}.gold_dim_products"
    )
else:
    raise ValueError("Choose customers or products")

# Prepare the initial historical records
scd_df = (
    source_df
    .withColumn("effective_start_date", current_timestamp())
    .withColumn("effective_end_date", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)

print("SCD Type 2 source prepared.")
print("Selected dimension:", dimension_name)


SCD Type 2 source prepared.
Selected dimension: products


In [0]:

# Define the target SCD Type 2 table
scd_table = f"{target_schema}.scd2_{dimension_name}"

# Create the table only if it does not already exist
if not spark.catalog.tableExists(scd_table):
    scd_df.write \
        .format("delta") \
        .mode("errorifexists") \
        .saveAsTable(scd_table)

    print("SCD Type 2 table created:", scd_table)
else:
    print("SCD Type 2 table already exists. Keeping existing history:", scd_table)



SCD Type 2 table already exists. Keeping existing history: adv_dev_westus3_demo_001.ecommerce_01.scd2_products


In [0]:

from pyspark.sql.functions import col, current_timestamp, lit

# Select the business key for the chosen dimension
key_column = "customer_id" if dimension_name == "customers" else "product_id"

source_df = spark.table(f"{target_schema}.gold_dim_{dimension_name}")
scd_table = f"{target_schema}.scd2_{dimension_name}"

# Compare source attributes with current SCD records
current_df = spark.table(scd_table).filter(col("is_current") == True)
tracked_columns = [c for c in source_df.columns if c != key_column]

comparison = " OR ".join(
    [f"NOT (s.`{c}` <=> t.`{c}`)" for c in tracked_columns]
)

changed_keys = (
    source_df.alias("s")
    .join(current_df.alias("t"), key_column, "inner")
    .filter(comparison if comparison else "false")
    .select(key_column)
    .distinct()
)

changed_count = changed_keys.count()

# Expire changed current records
if changed_count > 0:
    changed_keys.createOrReplaceTempView("changed_scd_keys")
    spark.sql(f"""
        MERGE INTO {scd_table} AS t
        USING changed_scd_keys AS s
        ON t.{key_column} = s.{key_column}
           AND t.is_current = true
        WHEN MATCHED THEN UPDATE SET
            t.effective_end_date = current_timestamp(),
            t.is_current = false
    """)

# Insert source records that have no current version
current_keys = (
    spark.table(scd_table)
    .filter(col("is_current") == True)
    .select(key_column)
    .distinct()
)

new_versions = (
    source_df.join(current_keys, key_column, "left_anti")
    .withColumn("effective_start_date", current_timestamp())
    .withColumn("effective_end_date", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)

insert_count = new_versions.count()

if insert_count > 0:
    new_versions.write.format("delta").mode("append").saveAsTable(scd_table)

print("Dimension:", dimension_name)
print("Changed existing records:", changed_count)
print("New current versions inserted:", insert_count)
print("SCD Type 2 processing completed.")




Dimension: products
Changed existing records: 0
New current versions inserted: 0
SCD Type 2 processing completed.


In [0]:

# Verify the selected SCD Type 2 dimension
display(
    spark.table(scd_table)
    .select(
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .limit(10)
)


effective_start_date,effective_end_date,is_current
2026-10-23T15:50:04.461Z,null,true
2026-10-23T15:50:04.461Z,null,true
2026-10-23T15:50:04.461Z,null,true
2026-10-23T15:50:04.461Z,null,true
2026-10-23T15:50:04.461Z,null,true
2026-10-23T15:50:04.461Z,null,true
2026-10-23T15:50:04.461Z,null,true
2026-10-23T15:50:04.461Z,null,true
2026-10-23T15:50:04.461Z,null,true
2026-10-23T15:50:04.461Z,null,true


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

key_column = "customer_id" if dimension_name == "customers" else "product_id"

scd_table = f"{target_schema}.scd2_{dimension_name}"
delta_table = DeltaTable.forName(spark, scd_table)

change_condition = " OR ".join([
    f"NOT (t.`{c}` <=> s.`{c}`)"
    for c in tracked_columns
])

delta_table.alias("t").merge(
    source_df.alias("s"),
    f"t.{key_column} = s.{key_column} AND t.is_current = true"
).whenMatchedUpdate(
    condition=change_condition,
    set={
        "effective_end_date": "current_timestamp()",
        "is_current": "false"
    }
).execute()

print("Dimension:", dimension_name)
print("Key column:", key_column)
print("Target table:", scd_table)
print("Changed versions marked as historical.")

Dimension: products
Key column: product_id
Target table: adv_dev_westus3_demo_001.ecommerce_01.scd2_products
Changed versions marked as historical.


In [0]:

from pyspark.sql.functions import current_timestamp, lit, col

key_column = "customer_id" if dimension_name == "customers" else "product_id"

current_df = (
    spark.table(scd_table)
    .filter(col("is_current") == True)
)

new_versions = (
    source_df.alias("s")
    .join(
        current_df.select(key_column),
        on=key_column,
        how="left_anti"
    )
    .withColumn("effective_start_date", current_timestamp())
    .withColumn("effective_end_date", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)

insert_count = new_versions.count()

if insert_count > 0:
    new_versions.write.format("delta").mode("append").saveAsTable(scd_table)

print("Dimension:", dimension_name)
print("Key column:", key_column)
print("New current versions inserted:", insert_count)


Dimension: products
Key column: product_id
New current versions inserted: 0


In [0]:


from pyspark.sql.functions import col

for name in ["customers", "products"]:
    df = spark.table(f"{target_schema}.scd2_{name}")
    key_column = "customer_id" if name == "customers" else "product_id"

    print(f"\nDimension: {name}")
    print("Total records:", df.count())
    print("Missing keys:", df.filter(col(key_column).isNull()).count())
    print("Current records:", df.filter(col("is_current") == True).count())
    print("Historical records:", df.filter(col("is_current") == False).count())



Dimension: customers
Total records: 99441
Missing keys: 0
Current records: 99441
Historical records: 0

Dimension: products
Total records: 32951
Missing keys: 0
Current records: 32951
Historical records: 0


In [0]:
spark.sql("""
SHOW TABLES IN adv_dev_westus3_demo_001.ecommerce_01
""").show(truncate=False)

+------------+----------------------------+-----------+
|database    |tableName                   |isTemporary|
+------------+----------------------------+-----------+
|ecommerce_01|fact_sales                  |false      |
|ecommerce_01|gold_daily_sales            |false      |
|ecommerce_01|gold_dim_customers          |false      |
|ecommerce_01|gold_dim_products           |false      |
|ecommerce_01|gold_fact_order_sales       |false      |
|ecommerce_01|gold_monthly_sales          |false      |
|ecommerce_01|gold_sales_by_state         |false      |
|ecommerce_01|gold_top_products_by_revenue|false      |
|ecommerce_01|rollback_test               |false      |
|ecommerce_01|scd2_customers              |false      |
|ecommerce_01|scd2_products               |false      |
|ecommerce_01|silver_customers            |false      |
|ecommerce_01|silver_order_items          |false      |
|ecommerce_01|silver_order_payments       |false      |
|ecommerce_01|silver_orders               |false

In [0]:
# Define the target SCD Type 2 table
scd_table = f"{target_schema}.scd2_{dimension_name}"

# Create the table only if it does not already exist
if not spark.catalog.tableExists(scd_table):
    scd_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(scd_table)

    print("SCD Type 2 table created:", scd_table)
else:
    print("SCD Type 2 table already exists. Keeping existing history:", scd_table)

SCD Type 2 table already exists. Keeping existing history: adv_dev_westus3_demo_001.ecommerce_01.scd2_products


In [0]:
spark.sql("""
SHOW TABLES IN adv_dev_westus3_demo_001.ecommerce_01
""").show(truncate=False)

+------------+----------------------------+-----------+
|database    |tableName                   |isTemporary|
+------------+----------------------------+-----------+
|ecommerce_01|fact_sales                  |false      |
|ecommerce_01|gold_daily_sales            |false      |
|ecommerce_01|gold_dim_customers          |false      |
|ecommerce_01|gold_dim_products           |false      |
|ecommerce_01|gold_fact_order_sales       |false      |
|ecommerce_01|gold_monthly_sales          |false      |
|ecommerce_01|gold_sales_by_state         |false      |
|ecommerce_01|gold_top_products_by_revenue|false      |
|ecommerce_01|rollback_test               |false      |
|ecommerce_01|scd2_customers              |false      |
|ecommerce_01|scd2_products               |false      |
|ecommerce_01|silver_customers            |false      |
|ecommerce_01|silver_order_items          |false      |
|ecommerce_01|silver_order_payments       |false      |
|ecommerce_01|silver_orders               |false